# AgroVerify Edge — Commodity Image Classifier
**MobileNetV3-Small → ONNX → TFLite INT8**

Trains a 10-class commodity classifier and exports it ready for on-device inference.

**Before starting:** Runtime → Change runtime type → **T4 GPU**

**Estimated time:** dataset ~90 min · training ~40 min · export ~5 min

**Classes:** maize, cassava, sorghum, rice, soy, groundnuts, yam, millet, cocoa, palm_oil

## 0 — Session variables  *(re-run this cell first after any session restart)*

In [ ]:
import os
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_URL      = 'https://github.com/basseyekpenyong/agroverify_edge_project.git'
WORK_DIR      = '/content/drive/MyDrive/agroverify_classifier'          # Drive root
REPO_DIR      = os.path.join(WORK_DIR, 'repo')                          # cloned repo
CLASSIFIER_DIR = os.path.join(REPO_DIR, 'ai-models', 'vision', 'commodity_classifier')

# Derived Drive-backed directories (symlinked into the repo below)
DATASET_DRIVE  = os.path.join(WORK_DIR, 'datasets')
CKPT_DRIVE     = os.path.join(WORK_DIR, 'checkpoints')
EXPORTS_DRIVE  = os.path.join(WORK_DIR, 'exports')

print('Paths configured:')
print(f'  WORK_DIR      : {WORK_DIR}')
print(f'  REPO_DIR      : {REPO_DIR}')
print(f'  CLASSIFIER_DIR: {CLASSIFIER_DIR}')

## 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(WORK_DIR, exist_ok=True)
print('Drive mounted. Working root:', WORK_DIR)

## 3 — Clone repo and install dependencies

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    !git clone "{REPO_URL}" "{REPO_DIR}"
else:
    print('Repo already cloned — pulling latest...')
    !git -C "{REPO_DIR}" pull

# Verify key files exist
for fname in ['download_dataset.py', 'train.py', 'export_tflite.py']:
    full = os.path.join(CLASSIFIER_DIR, fname)
    status = '✅' if os.path.exists(full) else '❌ MISSING'
    print(f'  {fname}: {status}')

os.chdir(CLASSIFIER_DIR)
print('\nWorking directory:', os.getcwd())

In [ ]:
# ── Private repo? Add a GitHub token here ──────────────────────────────────
# If the repo is private, re-run the clone cell above using:
#   REPO_URL = 'https://<YOUR_TOKEN>@github.com/basseyekpenyong/agroverify_edge_project.git'
# Generate a token at: GitHub → Settings → Developer Settings → Personal Access Tokens
# Scope needed: repo (read)
# ───────────────────────────────────────────────────────────────────────────

!pip install -q \
    "onnx>=1.16.0" \
    "onnxruntime>=1.18.0" \
    "onnx2tf>=1.20.0" \
    "icrawler>=0.6.6" \
    tqdm \
    Pillow

print('✅ Dependencies installed')

## 4 — Link Drive directories into repo  *(safe to re-run)*

In [ ]:
import os

def safe_symlink(drive_path: str, local_path: str, label: str) -> None:
    """Create symlink from local_path → drive_path. Handles stale links safely."""
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(local_path):
        if os.readlink(local_path) == drive_path:
            print(f'  {label}: already linked ✅')
            return
        os.unlink(local_path)   # stale link → remove and re-create
    if os.path.exists(local_path) and not os.path.islink(local_path):
        print(f'  {label}: real directory exists — skipping symlink')
        return
    os.symlink(drive_path, local_path)
    print(f'  {label}: linked to Drive ✅')

safe_symlink(DATASET_DRIVE,  os.path.join(CLASSIFIER_DIR, 'datasets'),    'datasets')
safe_symlink(CKPT_DRIVE,     os.path.join(CLASSIFIER_DIR, 'checkpoints'), 'checkpoints')
safe_symlink(EXPORTS_DRIVE,  os.path.join(CLASSIFIER_DIR, 'exports'),     'exports')

## 5 — Download Dataset
~600 images/class from iNaturalist + Bing. Skips classes already at target.

In [ ]:
# Check existing downloads before starting
from pathlib import Path
raw_dir = Path(CLASSIFIER_DIR) / 'datasets' / 'commodity_images' / 'raw'
if raw_dir.exists():
    print('Existing raw images:')
    for cls in sorted(raw_dir.iterdir()):
        if cls.is_dir():
            n = len(list(cls.glob('*.jpg')))
            bar = '█' * (n // 30)
            print(f'  {cls.name:12s} {bar} {n}')
else:
    print('No images yet — starting fresh download')

In [ ]:
os.chdir(CLASSIFIER_DIR)   # ensure cwd is correct after any restart
!python download_dataset.py --per-class 600

In [ ]:
# Verify split counts
from pathlib import Path
dataset_dir = Path(CLASSIFIER_DIR) / 'datasets' / 'commodity_images'
print(f'{"split":8s}  {"images":>8s}')
print('-' * 20)
for split in ['train', 'val', 'test']:
    split_dir = dataset_dir / split
    if not split_dir.exists():
        print(f'{split:8s}  not found ❌')
        continue
    total = sum(
        len(list(cls.glob('*.jpg')))
        for cls in split_dir.iterdir() if cls.is_dir()
    )
    print(f'{split:8s}  {total:>8d}')

## 6 — Train  (~40 min on T4)

In [ ]:
from pathlib import Path
ckpt = Path(CLASSIFIER_DIR) / 'checkpoints' / 'commodity_classifier.pt'
if ckpt.exists():
    print(f'Checkpoint found: {ckpt.stat().st_size / 1e6:.1f} MB — will overwrite only if new best accuracy found')
else:
    print('No checkpoint yet — starting fresh training')

os.chdir(CLASSIFIER_DIR)
!python train.py

## 7 — Evaluate on Test Set

In [ ]:
import torch, torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from pathlib import Path

os.chdir(CLASSIFIER_DIR)

NUM_CLASSES = 10
IMG_SIZE = 224
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.mobilenet_v3_small(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)
model.load_state_dict(torch.load('checkpoints/commodity_classifier.pt', map_location=device, weights_only=True))
model = model.to(device).eval()

test_ds = datasets.ImageFolder(
    'datasets/commodity_images/test',
    transform=transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
)
loader = DataLoader(test_ds, batch_size=32, num_workers=2)
idx_to_class = {v: k for k, v in test_ds.class_to_idx.items()}

correct = total = 0
class_correct = [0] * NUM_CLASSES
class_total   = [0] * NUM_CLASSES

with torch.no_grad():
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        preds = model(imgs).argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
        for p, l in zip(preds, lbls):
            class_correct[l] += (p == l).item()
            class_total[l]   += 1

print(f'Overall test accuracy: {correct/total*100:.2f}%  ({correct}/{total})')
print(f'\nPer-class accuracy:')
for i in range(NUM_CLASSES):
    if class_total[i]:
        acc = class_correct[i] / class_total[i] * 100
        flag = '⚠️' if acc < 75 else '✅'
        print(f'  {flag} {idx_to_class[i]:12s}: {acc:.1f}%  ({class_correct[i]}/{class_total[i]})')

## 8 — Export to TFLite INT8

In [ ]:
os.chdir(CLASSIFIER_DIR)
!python export_tflite.py

## 9 — Verify TFLite model

In [ ]:
import numpy as np, tensorflow as tf
from pathlib import Path

os.chdir(CLASSIFIER_DIR)
TFLITE_PATH = 'exports/commodity_classifier_int8.tflite'

if not Path(TFLITE_PATH).exists():
    print('❌ TFLite file not found — re-run Step 8')
else:
    interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]

    print('TFLite model info:')
    print(f'  Size  : {Path(TFLITE_PATH).stat().st_size / 1024:.0f} KB')
    print(f'  Input : shape={inp["shape"]}  dtype={inp["dtype"].__name__}')
    print(f'  Output: shape={out["shape"]}  dtype={out["dtype"].__name__}')

    dummy = np.random.rand(1, 224, 224, 3).astype(np.float32)
    interp.set_tensor(inp['index'], dummy)
    interp.invoke()
    scores = interp.get_tensor(out['index'])[0]
    CLASSES = ['maize','cassava','sorghum','rice','soy','groundnuts','yam','millet','cocoa','palm_oil']
    print(f'\n  Sanity inference: predicted "{CLASSES[np.argmax(scores)]}" ✅')

## 10 — Download to your computer
Copy the downloaded `.tflite` into `mobile-app/assets/models/` then run `flutter run`.

In [ ]:
from google.colab import files
import os
os.chdir(CLASSIFIER_DIR)

for f in ['exports/commodity_classifier_int8.tflite', 'exports/labels.txt']:
    from pathlib import Path
    if Path(f).exists():
        files.download(f)
        print(f'⬇️  Downloading {f}')
    else:
        print(f'❌ Not found: {f} — re-run Step 8')

print()
print('Place the .tflite in:')
print('  mobile-app/assets/models/commodity_classifier_int8.tflite')
print('Then: flutter run')

---
## Summary

| | |
|---|---|
| Dataset | ~6,000 images · 10 classes · 70/20/10 split |
| Model | MobileNetV3-Small · 30 epochs |
| Target accuracy | ≥ 85% top-1 |
| TFLite size | < 5 MB INT8 |
| Inference target | < 2s on mid-range Android |
| Persistence | All outputs saved to Google Drive `agroverify_classifier/` |